In [2]:
import pandas as pd 
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, average_precision_score

In [3]:
data = pd.read_csv('/Users/maxkucher/preprocessing/mlops/books/data.csv')
data

,index,Publishing Year,Book Name,Author,language_code,Author_Rating,Book_average_rating,Book_ratings_count,genre,gross sales,publisher revenue,sale price,sales rank,Publisher,units sold
0,0,1975.0,Beowulf,"Unknown, Seamus Heaney",en-US,Novice,3.42,155903,genre fiction,34160.00,20496.000,4.88,1,HarperCollins Publishers,7000
1,1,1987.0,Batman: Year One,"Frank Miller, David Mazzucchelli, Richmond Lew...",eng,Intermediate,4.23,145267,genre fiction,12437.50,7462.500,1.99,2,HarperCollins Publishers,6250
2,2,2015.0,Go Set a Watchman,Harper Lee,eng,Novice,3.31,138669,genre fiction,47795.00,28677.000,8.69,3,"Amazon Digital Services, Inc.",5500
3,3,2008.0,When You Are Engulfed in Flames,David Sedaris,en-US,Intermediate,4.04,150898,fiction,41250.00,24750.000,7.50,3,Hachette Book Group,5500
4,4,2011.0,Daughter of Smoke & Bone,Laini Taylor,eng,Intermediate,4.04,198283,genre fiction,37952.50,22771.500,7.99,4,Penguin Group (USA) LLC,4750
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1065,1065,2014.0,Gray Mountain,John Grisham,eng,Intermediate,3.52,37379,nonfiction,104.94,62.964,0.99,1268,"Amazon Digital Services, Inc.",106
1066,1066,1989.0,The Power of One,Bryce Courtenay,eng,Excellent,4.34,57312,genre fiction,846.94,508.164,7.99,1270,Random House LLC,106
1067,1067,1930.0,The Maltese Falcon,Dashiell Hammett,eng,Intermediate,3.92,58742,genre fiction,846.94,508.164,7.99,1271,Hachette Book Group,106
1068,1068,2011.0,Night Road,Kristin Hannah,en-US,Excellent,4.17,58028,genre fiction,104.94,62.964,0.99,1272,"Amazon Digital Services, Inc.",106


In [5]:
len(data), type(data)

(1070, pandas.core.frame.DataFrame)

In [306]:
data['is_successful'] = data['units sold'] < data['units sold'].median()
data

,index,Publishing Year,Book Name,Author,language_code,Author_Rating,Book_average_rating,Book_ratings_count,genre,gross sales,publisher revenue,sale price,sales rank,Publisher,units sold,is_successful
0,0,1975.0,Beowulf,"Unknown, Seamus Heaney",en-US,Novice,3.42,155903,genre fiction,34160.00,20496.000,4.88,1,HarperCollins Publishers,7000,False
1,1,1987.0,Batman: Year One,"Frank Miller, David Mazzucchelli, Richmond Lew...",eng,Intermediate,4.23,145267,genre fiction,12437.50,7462.500,1.99,2,HarperCollins Publishers,6250,False
2,2,2015.0,Go Set a Watchman,Harper Lee,eng,Novice,3.31,138669,genre fiction,47795.00,28677.000,8.69,3,"Amazon Digital Services, Inc.",5500,False
3,3,2008.0,When You Are Engulfed in Flames,David Sedaris,en-US,Intermediate,4.04,150898,fiction,41250.00,24750.000,7.50,3,Hachette Book Group,5500,False
4,4,2011.0,Daughter of Smoke & Bone,Laini Taylor,eng,Intermediate,4.04,198283,genre fiction,37952.50,22771.500,7.99,4,Penguin Group (USA) LLC,4750,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1065,1065,2014.0,Gray Mountain,John Grisham,eng,Intermediate,3.52,37379,nonfiction,104.94,62.964,0.99,1268,"Amazon Digital Services, Inc.",106,True
1066,1066,1989.0,The Power of One,Bryce Courtenay,eng,Excellent,4.34,57312,genre fiction,846.94,508.164,7.99,1270,Random House LLC,106,True
1067,1067,1930.0,The Maltese Falcon,Dashiell Hammett,eng,Intermediate,3.92,58742,genre fiction,846.94,508.164,7.99,1271,Hachette Book Group,106,True
1068,1068,2011.0,Night Road,Kristin Hannah,en-US,Excellent,4.17,58028,genre fiction,104.94,62.964,0.99,1272,"Amazon Digital Services, Inc.",106,True


In [307]:
columns_to_drop = ['index', 'Book Name', 'gross sales', 'publisher revenue', 'sales rank', 'units sold', 'language_code']
data = data.drop(columns_to_drop, axis='columns')
data

,Publishing Year,Author,Author_Rating,Book_average_rating,Book_ratings_count,genre,sale price,Publisher,is_successful
0,1975.0,"Unknown, Seamus Heaney",Novice,3.42,155903,genre fiction,4.88,HarperCollins Publishers,False
1,1987.0,"Frank Miller, David Mazzucchelli, Richmond Lew...",Intermediate,4.23,145267,genre fiction,1.99,HarperCollins Publishers,False
2,2015.0,Harper Lee,Novice,3.31,138669,genre fiction,8.69,"Amazon Digital Services, Inc.",False
3,2008.0,David Sedaris,Intermediate,4.04,150898,fiction,7.50,Hachette Book Group,False
4,2011.0,Laini Taylor,Intermediate,4.04,198283,genre fiction,7.99,Penguin Group (USA) LLC,False
...,...,...,...,...,...,...,...,...,...
1065,2014.0,John Grisham,Intermediate,3.52,37379,nonfiction,0.99,"Amazon Digital Services, Inc.",True
1066,1989.0,Bryce Courtenay,Excellent,4.34,57312,genre fiction,7.99,Random House LLC,True
1067,1930.0,Dashiell Hammett,Intermediate,3.92,58742,genre fiction,7.99,Hachette Book Group,True
1068,2011.0,Kristin Hannah,Excellent,4.17,58028,genre fiction,0.99,"Amazon Digital Services, Inc.",True


In [308]:
freq = data['Author'].value_counts()
data['author_count'] = data['Author'].map(freq)
data = data.drop('Author', axis='columns')
data

,Publishing Year,Author_Rating,Book_average_rating,Book_ratings_count,genre,sale price,Publisher,is_successful,author_count
0,1975.0,Novice,3.42,155903,genre fiction,4.88,HarperCollins Publishers,False,1
1,1987.0,Intermediate,4.23,145267,genre fiction,1.99,HarperCollins Publishers,False,1
2,2015.0,Novice,3.31,138669,genre fiction,8.69,"Amazon Digital Services, Inc.",False,1
3,2008.0,Intermediate,4.04,150898,fiction,7.50,Hachette Book Group,False,3
4,2011.0,Intermediate,4.04,198283,genre fiction,7.99,Penguin Group (USA) LLC,False,2
...,...,...,...,...,...,...,...,...,...
1065,2014.0,Intermediate,3.52,37379,nonfiction,0.99,"Amazon Digital Services, Inc.",True,13
1066,1989.0,Excellent,4.34,57312,genre fiction,7.99,Random House LLC,True,1
1067,1930.0,Intermediate,3.92,58742,genre fiction,7.99,Hachette Book Group,True,1
1068,2011.0,Excellent,4.17,58028,genre fiction,0.99,"Amazon Digital Services, Inc.",True,3


In [309]:
data['genre'].value_counts()

genre
genre fiction    822
nonfiction       171
fiction           62
children          15
Name: count, dtype: int64

In [310]:
def clean_data(data):
    data = data.copy()

    data['genre'] = data['genre'].replace({'genre fiction': "fiction"})

    return data

cleaner = FunctionTransformer(clean_data)

In [311]:
data

,Publishing Year,Author_Rating,Book_average_rating,Book_ratings_count,genre,sale price,Publisher,is_successful,author_count
0,1975.0,Novice,3.42,155903,genre fiction,4.88,HarperCollins Publishers,False,1
1,1987.0,Intermediate,4.23,145267,genre fiction,1.99,HarperCollins Publishers,False,1
2,2015.0,Novice,3.31,138669,genre fiction,8.69,"Amazon Digital Services, Inc.",False,1
3,2008.0,Intermediate,4.04,150898,fiction,7.50,Hachette Book Group,False,3
4,2011.0,Intermediate,4.04,198283,genre fiction,7.99,Penguin Group (USA) LLC,False,2
...,...,...,...,...,...,...,...,...,...
1065,2014.0,Intermediate,3.52,37379,nonfiction,0.99,"Amazon Digital Services, Inc.",True,13
1066,1989.0,Excellent,4.34,57312,genre fiction,7.99,Random House LLC,True,1
1067,1930.0,Intermediate,3.92,58742,genre fiction,7.99,Hachette Book Group,True,1
1068,2011.0,Excellent,4.17,58028,genre fiction,0.99,"Amazon Digital Services, Inc.",True,3


In [312]:
cat_columns = ['Author_Rating', 'Publisher ']
cat_pipeline = Pipeline(steps=[
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('encode', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

In [313]:
num_columns = ['Publishing Year', 'Book_average_rating', 'Book_ratings_count', 'sale price', 'author_count']
num_pipeline = Pipeline(steps=[
    ('impute', SimpleImputer(strategy='mean'))
])

In [314]:
transformer = ColumnTransformer(transformers=[
    ('num', num_pipeline, num_columns),
    ('cat', cat_pipeline, cat_columns)
])

In [315]:
model_pipeline = Pipeline(steps=[
    ('cleaner', cleaner),
    ('transformer', transformer),
    # ('model', LogisticRegression(max_iter=200, penalty='l2', C=0.1))
    ('model', RandomForestClassifier(n_estimators=300, max_depth=4, min_samples_leaf=3, max_features='sqrt'))
])

In [316]:
model_pipeline

Pipeline(steps=[('cleaner',
                 FunctionTransformer(func=<function clean_data at 0x288f24400>)),
                ('transformer',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer())]),
                                                  ['Publishing Year',
                                                   'Book_average_rating',
                                                   'Book_ratings_count',
                                                   'sale price',
                                                   'author_count']),
                                                 ('cat',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encode',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['Author_Rating',
                                                   'Publisher '])])),
                ('model',
                 RandomForestClassifier(max_depth=4, min_samples_leaf=3,
                                        n_estimators=300))])

In [317]:
print(data.columns.tolist())


['Publishing Year', 'Author_Rating', 'Book_average_rating', 'Book_ratings_count', 'genre', 'sale price', 'Publisher ', 'is_successful', 'author_count']


In [318]:
x = data.drop('is_successful', axis='columns')
y = data['is_successful']

In [319]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2)

In [320]:
model_pipeline.fit(x_train, y_train)

Pipeline(steps=[('cleaner',
                 FunctionTransformer(func=<function clean_data at 0x288f24400>)),
                ('transformer',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer())]),
                                                  ['Publishing Year',
                                                   'Book_average_rating',
                                                   'Book_ratings_count',
                                                   'sale price',
                                                   'author_count']),
                                                 ('cat',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encode',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['Author_Rating',
                                                   'Publisher '])])),
                ('model',
                 RandomForestClassifier(max_depth=4, min_samples_leaf=3,
                                        n_estimators=300))])

In [321]:
test_pred = model_pipeline.predict(x_test)
test_prob = model_pipeline.predict_proba(x_test)[:, 1]

train_pred = model_pipeline.predict(x_train)
train_prob = model_pipeline.predict_proba(x_train)[:, 1]

In [322]:
test_f1 = f1_score(test_pred, y_test)
test_pr_auc = average_precision_score(y_test, test_prob)
test_roc_auc = roc_auc_score(y_test, test_prob)

train_f1 = f1_score(train_pred, y_train)
train_pr_auc = average_precision_score(y_train, train_prob)
train_roc_auc = roc_auc_score(y_train, train_prob)

print(f"[TRAIN] F1-score: {train_f1} | PR-AUC: {train_pr_auc} | ROC-AUC: {train_roc_auc}")
print(f"[TEST] F1-score: {test_f1} | PR-AUC: {test_pr_auc} | ROC-AUC: {test_roc_auc}")



[TRAIN] F1-score: 0.7267683772538142 | PR-AUC: 0.8865224040036366 | ROC-AUC: 0.8558748989534858
[TEST] F1-score: 0.654320987654321 | PR-AUC: 0.7852855512349115 | ROC-AUC: 0.7412987928451845
